### Markus Hoehn and Leonard Collomb

# Wildfire Damage Visualizations

This notebook builds a clean set of **standalone PNG map exports** for the **Palisades** fire using the **H3 resolution 10 context table** produced by the data extraction notebook.

expects: `Palisades_context_hex_data.csv` and `Palisades_Public_View.geojson` in the directory


In [1]:
!pip install -q h3pandas contextily rasterio requests whitebox

import os
from pathlib import Path
from zipfile import ZipFile

import contextily as ctx
import geopandas as gpd
import h3pandas
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import rasterio
import requests

from matplotlib.colors import TwoSlopeNorm
from rasterio.enums import Resampling
from rasterio.merge import merge
from rasterio.windows import Window, from_bounds
from rasterio.warp import transform_bounds
from shapely import wkt
from whitebox.whitebox_tools import WhiteboxTools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.0 MB/s eta 0:00:00


In [2]:
fire_name = "Palisades"
RES = 10
OUTPUT_DIR = Path("palisades_visual_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def first_existing(paths):
    for path in paths:
        if os.path.exists(path):
            return path
    raise FileNotFoundError("Could not find any of these paths:\n" + "\n".join(paths))

context_path = first_existing([
    f"{fire_name}_context_hex_data.csv",
    f"{fire_name}_context_hex_data-1.csv",
    f"{fire_name}_context_data.csv",
    f"/mnt/data/{fire_name}_context_hex_data.csv",
    f"/mnt/data/{fire_name}_context_hex_data-1.csv",
    f"/mnt/data/{fire_name}_context_data.csv",
])

geojson_path = first_existing([
    f"{fire_name}_Public_View.geojson",
    f"/mnt/data/{fire_name}_Public_View.geojson",
])

context_df = pd.read_csv(context_path, low_memory=False)
ghex = gpd.GeoDataFrame(
    context_df.drop(columns=["geometry"]),
    geometry=context_df["geometry"].apply(wkt.loads),
    crs="EPSG:4326",
)

if "h3_10" in ghex.columns:
    ghex["h3_10"] = ghex["h3_10"].astype(str)
    ghex = ghex.set_index("h3_10")
else:
    ghex.index = context_df.iloc[:, 0].astype(str)
    ghex.index.name = "h3_10"

gdf_raw = gpd.read_file(geojson_path).to_crs(4326)

print("context_path:", context_path)
print("geojson_path:", geojson_path)
print("context rows:", len(ghex))
print("raw inspection rows:", len(gdf_raw))

context_path: Palisades_context_hex_data.csv
geojson_path: Palisades_Public_View.geojson
context rows: 34902
raw inspection rows: 12081


In [3]:
# Rebuild the Palisades actual_damage_rate in the same style as the modeling notebook.
damage_dummies = pd.get_dummies(gdf_raw["DAMAGE"], dtype=int)
gdf_damage = gdf_raw[["geometry"]].join(damage_dummies)

reconstructed_hex = gdf_damage.h3.geo_to_h3_aggregate(RES)
reconstructed_hex.index = reconstructed_hex.index.astype(str)
reconstructed_hex.index.name = "h3_10"

damage_cols = [col for col in damage_dummies.columns if col != "No Damage"]
reconstructed_hex["record_count"] = reconstructed_hex[damage_dummies.columns].sum(axis=1)
reconstructed_hex["actual_damage_rate"] = (
    reconstructed_hex[damage_cols].sum(axis=1) / reconstructed_hex["record_count"]
)

ghex = ghex.join(
    reconstructed_hex[["record_count", "actual_damage_rate"]],
    how="left",
)

print(ghex[["record_count", "actual_damage_rate"]].dropna().head())

                 record_count  actual_damage_rate
h3_10                                            
8a29a1800427fff           2.0                 0.0
8a29a1800507fff           3.0                 0.0
8a29a180050ffff           1.0                 0.0
8a29a1800517fff           3.0                 0.0
8a29a180052ffff           4.0                 0.0


In [4]:
def expand_bounds(bounds, frac):
    minx, miny, maxx, maxy = bounds
    dx = frac * (maxx - minx)
    dy = frac * (maxy - miny)
    return (minx - dx, miny - dy, maxx + dx, maxy + dy)

ghex_3857 = ghex.to_crs(epsg=3857)
gdf_3857 = gdf_raw.to_crs(epsg=3857)

plot_bounds_3857 = tuple(gdf_3857.total_bounds)
evt_bounds_3857 = plot_bounds_3857

plot_bounds_4326 = (
    gpd.GeoSeries([gdf_3857.unary_union], crs="EPSG:3857")
    .to_crs(4326)
    .total_bounds
)

# Read rasters from a larger box first, then crop every saved image back to the same final bounds.
buffered_fetch_bounds_4326 = expand_bounds(plot_bounds_4326, 0.50)

print("plot_bounds_3857:", plot_bounds_3857)
print("buffered_fetch_bounds_4326:", buffered_fetch_bounds_4326)

def set_axis_bounds(ax, bounds):
    xmin, ymin, xmax, ymax = bounds
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal")

def save_plain_contextily(bounds_3857, filename, zoom=14):
    fig, ax = plt.subplots(figsize=(8, 8))
    set_axis_bounds(ax, bounds_3857)
    ctx.add_basemap(
        ax,
        source=ctx.providers.Esri.WorldStreetMap,
        zoom=zoom,
        attribution=False,
    )
    set_axis_bounds(ax, bounds_3857)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    print(filename)

def save_numeric_hex_map(
    gdf_3857,
    column,
    filename,
    bounds_3857,
    cmap="inferno",
    vmin=None,
    vmax=None,
    norm=None,
    add_basemap=False,
    alpha=1.0,
    zoom=14,
):
    plot_df = gdf_3857[gdf_3857[column].notna()].copy()

    fig, ax = plt.subplots(figsize=(8, 8))
    plot_df.plot(
        column=column,
        cmap=cmap,
        ax=ax,
        linewidth=0,
        edgecolor="none",
        vmin=vmin,
        vmax=vmax,
        norm=norm,
        alpha=alpha,
        antialiased=False,
        zorder=2,
    )

    if add_basemap:
        ctx.add_basemap(
            ax,
            source=ctx.providers.Esri.WorldStreetMap,
            zoom=zoom,
            attribution=False,
        )

    set_axis_bounds(ax, bounds_3857)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    print(filename)


def save_categorical_hex_map(
    gdf_3857,
    column,
    filename,
    bounds_3857,
    color_map,
    missing_color="#d9d9d9",
    legend_title=None,
    pretty_values=None,
    category_order=None,
):
    vals = gdf_3857[column].fillna("Missing").astype(str)
    colors = vals.map(lambda x: color_map.get(x, missing_color))

    fig, ax = plt.subplots(figsize=(8, 8))
    gdf_3857.plot(
        color=colors,
        ax=ax,
        linewidth=0,
        edgecolor="none",
        antialiased=False,
        zorder=2,
    )
    set_axis_bounds(ax, bounds_3857)
    ax.set_axis_off()

    present_cats = list(pd.Index(vals.unique()))
    if category_order is not None:
        cats = [cat for cat in category_order if cat in present_cats]
        cats += [cat for cat in present_cats if cat not in cats]
    else:
        cats = sorted(present_cats)

    handles = [
        mpatches.Patch(
            color=color_map.get(cat, missing_color),
            label=(pretty_values or {}).get(cat, str(cat).replace("_", " ").title()),
        )
        for cat in cats
    ]

    if handles:
        ax.legend(
            handles=handles,
            title=legend_title,
            loc="lower left",
            fontsize=8,
            title_fontsize=9,
            frameon=True,
            borderpad=0.3,
            labelspacing=0.25,
            handlelength=1.2,
            handletextpad=0.4,
        )

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    print(filename)

def streamed_download(url, out_path, chunk_size=1024 * 1024):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        return out_path

    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return out_path

def ensure_zip_extracted(zip_url, base_dir):
    zip_path = Path(base_dir) / Path(zip_url).name
    extract_dir = Path(base_dir) / Path(zip_url).stem

    if not zip_path.exists():
        print(f"Downloading {zip_url} ...")
        streamed_download(zip_url, zip_path)

    if not extract_dir.exists():
        print(f"Extracting {zip_path.name} ...")
        extract_dir.mkdir(parents=True, exist_ok=True)
        with ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)

    return extract_dir

def find_first_file(root_dir, suffix, extra_pred=None):
    root_dir = Path(root_dir)
    for path in root_dir.rglob(f"*{suffix}"):
        if extra_pred is None or extra_pred(path):
            return path
    raise FileNotFoundError(f"Could not find a {suffix} under {root_dir}")

def find_landfire_legend_csv(extract_dir):
    extract_dir = Path(extract_dir)
    candidates = sorted(extract_dir.rglob("*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No legend CSV found under {extract_dir}")

    preferred = [
        path for path in candidates
        if "CSV_Data" in str(path.parent) or "csv_data" in str(path.parent).lower()
    ]
    if preferred:
        return preferred[0]
    return candidates[0]

def read_landfire_legend(csv_path):
    legend = pd.read_csv(csv_path)
    legend.columns = legend.columns.str.upper().str.strip()
    return legend

def build_rgb_lookup_from_legend(legend):
    value_key = pd.to_numeric(legend["VALUE"], errors="coerce")

    if {"RED", "GREEN", "BLUE"}.issubset(legend.columns):
        rgb = legend[["RED", "GREEN", "BLUE"]].copy().astype(float)
        if rgb.max().max() <= 1.0:
            rgb = (255 * rgb).round().astype(int)
    elif {"R", "G", "B"}.issubset(legend.columns):
        rgb = legend[["R", "G", "B"]].copy().astype(float)
        if rgb.max().max() <= 1.0:
            rgb = (255 * rgb).round().astype(int)
        else:
            rgb = rgb.round().astype(int)
    else:
        raise ValueError("Legend did not contain RGB columns.")

    rgb_lookup = {
        int(v): (row[0] / 255.0, row[1] / 255.0, row[2] / 255.0)
        for v, row in zip(value_key, rgb.to_numpy())
        if pd.notna(v)
    }
    return rgb_lookup

def save_official_landfire_hex_map(gdf_3857, value_col, legend, filename, bounds_3857, missing_color="#d9d9d9"):
    rgb_lookup = build_rgb_lookup_from_legend(legend)
    vals = pd.to_numeric(gdf_3857[value_col], errors="coerce")
    colors = vals.map(lambda x: rgb_lookup.get(int(x), missing_color) if pd.notna(x) else missing_color)

    fig, ax = plt.subplots(figsize=(8, 8))
    gdf_3857.plot(
        color=colors,
        ax=ax,
        linewidth=0,
        edgecolor="none",
        antialiased=False,
    )
    set_axis_bounds(ax, bounds_3857)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    print(filename)


plot_bounds_3857: (np.float64(-13212054.236100001), np.float64(4032803.6024999996), np.float64(-13191239.581300013), np.float64(4046112.086099992))
buffered_fetch_bounds_4326: (np.float64(-118.7793931617495), np.float64(33.9802857139714), np.float64(-118.40543071093946), np.float64(34.17832684345019))


In [5]:
save_plain_contextily(
    plot_bounds_3857,
    OUTPUT_DIR / "palisades_plain_contextily.png",
)

save_numeric_hex_map(
    ghex_3857,
    "actual_damage_rate",
    OUTPUT_DIR / "palisades_actual_damage_rate.png",
    plot_bounds_3857,
    cmap="inferno",
    vmin=0,
    vmax=1,
    add_basemap=True,
    alpha=0.6,
)

palisades_visual_outputs/palisades_plain_contextily.png
palisades_visual_outputs/palisades_actual_damage_rate.png


In [6]:
category_palettes = {
    "evt_lifeform": {
        "Developed": "#8c8c8c",
        "Herbaceous": "#7fc97f",
        "Shrubland": "#fdc086",
        "Tree": "#386cb0",
        "Sparse": "#f2f2f2",
        "Barren": "#f2f2f2",
        "Agriculture": "#ffff99",
        "Water": "#80b1d3",
        "Missing": "#d9d9d9",
    },
    "fuel_family": {
        "GR": "#7fc97f",
        "GS": "#beaed4",
        "SH": "#fdc086",
        "TU": "#66c2a5",
        "TL": "#8c6d31",
        "SB": "#984ea3",
        "NB": "#8c8c8c",
        "Missing": "#d9d9d9",
        "None": "#f2f2f2",
    },
    "dominant_fuel": {
        "grass": "#7fc97f",
        "shrub": "#fdc086",
        "litter": "#8c6d31",
        "slash": "#984ea3",
        "nonburnable": "#8c8c8c",
        "Missing": "#d9d9d9",
        "None": "#f2f2f2",
    },
    "expected_spread": {
        "low": "#d9f0a3",
        "moderate": "#addd8e",
        "high": "#78c679",
        "very_high": "#31a354",
        "extreme": "#006837",
        "Missing": "#d9d9d9",
        "None": "#f2f2f2",
    },
    "expected_flame": {
        "low": "#fee8c8",
        "moderate": "#fdbb84",
        "high": "#fc8d59",
        "very_high": "#e34a33",
        "extreme": "#b30000",
        "Missing": "#d9d9d9",
        "None": "#f2f2f2",
    },
}

pretty_titles = {
    "evt_lifeform": "EVT Lifeform",
    "fuel_family": "Fuel Family",
    "dominant_fuel": "Dominant Fuel",
    "expected_spread": "Expected Spread",
    "expected_flame": "Expected Flame",
}

pretty_values = {
    "evt_lifeform": {
        "Developed": "Developed",
        "Herbaceous": "Herbaceous",
        "Shrubland": "Shrubland",
        "Tree": "Tree",
        "Sparse": "Sparse",
        "Barren": "Barren",
        "Agriculture": "Agriculture",
        "Water": "Water",
        "Missing": "Missing",
    },
    "fuel_family": {
        "GR": "Grass",
        "GS": "Grass-Shrub",
        "SH": "Shrub",
        "TU": "Timber-Understory",
        "TL": "Timber-Litter",
        "SB": "Slash-Blowdown",
        "NB": "Non-Burnable",
        "Missing": "Missing",
        "None": "None",
    },
    "dominant_fuel": {
        "grass": "Grass",
        "shrub": "Shrub",
        "litter": "Litter",
        "slash": "Slash",
        "nonburnable": "Non-Burnable",
        "Missing": "Missing",
        "None": "None",
    },
    "expected_spread": {
        "low": "Low",
        "moderate": "Moderate",
        "high": "High",
        "very_high": "Very High",
        "extreme": "Extreme",
        "Missing": "Missing",
        "None": "None",
    },
    "expected_flame": {
        "low": "Low",
        "moderate": "Moderate",
        "high": "High",
        "very_high": "Very High",
        "extreme": "Extreme",
        "Missing": "Missing",
        "None": "None",
    },
}

category_orders = {
    "evt_lifeform": [
        "Developed", "Herbaceous", "Shrubland", "Tree", "Sparse",
        "Barren", "Agriculture", "Water", "Missing"
    ],
    "fuel_family": ["GR", "GS", "NB", "SH", "TL", "TU", "SB", "Missing", "None"],
    "dominant_fuel": ["grass", "litter", "nonburnable", "shrub", "slash", "Missing", "None"],
    "expected_spread": ["high", "low", "moderate", "very_high", "extreme", "Missing", "None"],
    "expected_flame": ["high", "low", "moderate", "very_high", "extreme", "Missing", "None"],
}

model_cat_cols = [
    "evt_lifeform",
    "fuel_family",
    "dominant_fuel",
    "expected_spread",
    "expected_flame",
]

for col in model_cat_cols:
    if col not in ghex_3857.columns:
        print(f"Skipping {col}: column not found.")
        continue

    save_categorical_hex_map(
        ghex_3857,
        col,
        OUTPUT_DIR / f"palisades_{col}.png",
        plot_bounds_3857,
        color_map=category_palettes[col],
        legend_title=pretty_titles[col],
        pretty_values=pretty_values[col],
        category_order=category_orders[col],
    )

palisades_visual_outputs/palisades_evt_lifeform.png
palisades_visual_outputs/palisades_fuel_family.png
palisades_visual_outputs/palisades_dominant_fuel.png
palisades_visual_outputs/palisades_expected_spread.png
palisades_visual_outputs/palisades_expected_flame.png


In [7]:
landfire_dir = Path(f"{fire_name}_landfire")
landfire_dir.mkdir(exist_ok=True)

print("Using H3-10 values already stored in the context table for EVC / EVH / FBFM40-style hex maps.")
print("Skipping the large EVC / EVH / FBFM40 LANDFIRE downloads here.")
print("Only DEM assets and the EVT LANDFIRE zip are still needed later in this notebook.")


Using H3-10 values already stored in the context table for EVC / EVH / FBFM40-style hex maps.
Skipping the large EVC / EVH / FBFM40 LANDFIRE downloads here.
Only DEM assets and the EVT LANDFIRE zip are still needed later in this notebook.


In [8]:
dem_dir = Path(f"{fire_name}_dem_tiles")
dem_work_dir = Path(f"{fire_name}_dem_work")
dem_dir.mkdir(exist_ok=True)
dem_work_dir.mkdir(exist_ok=True)

dem_manifest_url = (
    f"https://prd-tnm.s3.amazonaws.com/"
    f"CA_FireImpactZone_2025_PRELIMINARY/{fire_name}/"
    f"bare_earth/be_rasters/0_file_download_links.txt"
)

target_dem_res = 8
big_dem_path = dem_work_dir / f"{fire_name}_big_dem.tif"
resampled_dem_path = dem_work_dir / f"{fire_name}_dem_resampled_{target_dem_res}m.tif"

slope_file = dem_work_dir / f"{fire_name}_slope_deg.tif"
aspect_file = dem_work_dir / f"{fire_name}_aspect_deg.tif"
tri_file = dem_work_dir / f"{fire_name}_tri.tif"
plan_file = dem_work_dir / f"{fire_name}_plan_curvature.tif"
profile_file = dem_work_dir / f"{fire_name}_profile_curvature.tif"
tpi_small_file = dem_work_dir / f"{fire_name}_tpi_small.tif"
tpi_large_file = dem_work_dir / f"{fire_name}_tpi_large.tif"
sca_file = dem_work_dir / f"{fire_name}_sca.tif"
twi_file = dem_work_dir / f"{fire_name}_twi.tif"
breached_dem_file = dem_work_dir / "breached_dem.tif"

if not resampled_dem_path.exists():
    print("Downloading DEM manifest...")
    dem_manifest_text = requests.get(dem_manifest_url, timeout=120).text
    dem_urls = [
        line.strip()
        for line in dem_manifest_text.splitlines()
        if line.strip().endswith(".tif")
    ]
    if not dem_urls:
        raise RuntimeError(f"No DEM tile URLs found at {dem_manifest_url}")

    print(f"Downloading {len(dem_urls)} DEM tiles (missing files only)...")
    for i, url in enumerate(dem_urls, start=1):
        out_path = dem_dir / os.path.basename(url)
        if not out_path.exists():
            r = requests.get(url, timeout=300)
            r.raise_for_status()
            with open(out_path, "wb") as f:
                f.write(r.content)
        if i % 50 == 0 or i == len(dem_urls):
            print(f"  {i}/{len(dem_urls)}")

    dem_tile_paths = sorted(dem_dir.glob("*.tif"))
    if not dem_tile_paths:
        raise RuntimeError("No DEM tiles downloaded.")

    print("Merging DEM tiles...")
    srcs = [rasterio.open(path) for path in dem_tile_paths]
    merged_arr, merged_transform = merge(srcs)

    big_profile = srcs[0].profile | {
        "height": merged_arr.shape[1],
        "width": merged_arr.shape[2],
        "transform": merged_transform,
        "count": 1,
    }

    with rasterio.open(big_dem_path, "w", **big_profile) as dst:
        dst.write(merged_arr)

    for src in srcs:
        src.close()

    print("Resampling DEM...")
    with rasterio.open(big_dem_path) as src:
        xres, yres = map(abs, src.res)
        out_height = int(round(src.height * yres / target_dem_res))
        out_width = int(round(src.width * xres / target_dem_res))

        dem = src.read(
            1,
            out_shape=(out_height, out_width),
            resampling=Resampling.bilinear,
            masked=True,
        )

        transform = src.transform * src.transform.scale(
            src.width / out_width,
            src.height / out_height,
        )

        nodata = src.nodata if src.nodata is not None else -9999.0

        out_profile = src.profile | {
            "height": out_height,
            "width": out_width,
            "transform": transform,
            "count": 1,
            "dtype": "float32",
            "nodata": nodata,
        }

        with rasterio.open(resampled_dem_path, "w", **out_profile) as dst:
            dst.write(dem.filled(nodata).astype("float32"), 1)

terrain_outputs = {
    "elevation": resampled_dem_path,
    "slope_deg": slope_file,
    "tri": tri_file,
    "plan_curvature": plan_file,
    "profile_curvature": profile_file,
    "tpi_small": tpi_small_file,
    "tpi_large": tpi_large_file,
    "twi": twi_file,
}

missing_outputs = [path for path in terrain_outputs.values() if not path.exists()]
if missing_outputs:
    print("Computing derived terrain rasters with WhiteboxTools...")
    wbt = WhiteboxTools()
    wbt.set_verbose_mode(False)
    wbt.set_working_dir(str(dem_work_dir))

    analysis_dem_file = resampled_dem_path.name

    if not slope_file.exists():
        wbt.slope(dem=analysis_dem_file, output=slope_file.name, units="degrees")
    if not aspect_file.exists():
        wbt.aspect(dem=analysis_dem_file, output=aspect_file.name)
    if not tri_file.exists():
        wbt.ruggedness_index(dem=analysis_dem_file, output=tri_file.name)
    if not plan_file.exists():
        wbt.plan_curvature(dem=analysis_dem_file, output=plan_file.name)
    if not profile_file.exists():
        wbt.profile_curvature(dem=analysis_dem_file, output=profile_file.name)
    if not tpi_small_file.exists():
        wbt.diff_from_mean_elev(
            dem=analysis_dem_file,
            output=tpi_small_file.name,
            filterx=11,
            filtery=11,
        )
    if not tpi_large_file.exists():
        wbt.diff_from_mean_elev(
            dem=analysis_dem_file,
            output=tpi_large_file.name,
            filterx=31,
            filtery=31,
        )
    if not twi_file.exists():
        if not breached_dem_file.exists():
            wbt.breach_depressions(dem=analysis_dem_file, output=breached_dem_file.name)
        if not sca_file.exists():
            wbt.d8_flow_accumulation(
                i=breached_dem_file.name,
                output=sca_file.name,
                out_type="specific contributing area",
            )
        wbt.wetness_index(
            sca=sca_file.name,
            slope=slope_file.name,
            output=twi_file.name,
        )

def read_buffered_downsampled(path, out_hw=None, max_dim=1400, resampling=Resampling.bilinear):
    with rasterio.open(path) as src:
        bounds_src = transform_bounds(
            "EPSG:4326",
            src.crs,
            *buffered_fetch_bounds_4326,
            densify_pts=21,
        )
        window = from_bounds(*bounds_src, transform=src.transform)
        window = window.intersection(Window(0, 0, src.width, src.height))

        if out_hw is None:
            scale = max(window.height, window.width) / max_dim
            scale = max(scale, 1)
            out_h = max(1, int(window.height / scale))
            out_w = max(1, int(window.width / scale))
        else:
            out_h, out_w = out_hw

        arr = src.read(
            1,
            window=window,
            out_shape=(out_h, out_w),
            resampling=resampling,
            masked=True,
        ).astype("float32")

        left, bottom, right, top = rasterio.transform.array_bounds(
            out_h,
            out_w,
            src.window_transform(window),
        )
        extent_3857 = transform_bounds(src.crs, "EPSG:3857", left, bottom, right, top, densify_pts=21)

    return np.ma.filled(arr, np.nan), extent_3857

def hillshade(elevation, azimuth=315, altitude=45):
    elev = elevation.copy()
    finite = np.isfinite(elev)
    fill_val = np.nanmedian(elev[finite]) if finite.any() else 0.0
    elev[~finite] = fill_val

    dy, dx = np.gradient(elev)
    slope = np.pi / 2 - np.arctan(np.sqrt(dx * dx + dy * dy))
    aspect = np.arctan2(-dx, dy)

    az = np.deg2rad(azimuth)
    alt = np.deg2rad(altitude)

    hs = (
        np.sin(alt) * np.sin(slope)
        + np.cos(alt) * np.cos(slope) * np.cos(az - aspect)
    )
    hs = np.clip(hs, 0, 1)
    hs[~finite] = np.nan
    return hs

def robust_diverging_norm(arr, pct=98):
    vals = arr[np.isfinite(arr)]
    if vals.size == 0:
        return TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
    lim = np.nanpercentile(np.abs(vals), pct)
    if not np.isfinite(lim) or lim == 0:
        lim = 1.0
    return TwoSlopeNorm(vmin=-lim, vcenter=0, vmax=lim)

def save_hillshaded_raster(arr, extent_3857, hillshade_arr, filename, bounds_3857, cmap="RdBu_r", norm=None, alpha=0.8):
    fig, ax = plt.subplots(figsize=(8, 8))

    hs_cmap = plt.cm.get_cmap("Greys").copy()
    hs_cmap.set_bad((1, 1, 1, 0))
    ax.imshow(
        hillshade_arr,
        cmap=hs_cmap,
        interpolation="nearest",
        origin="upper",
        extent=extent_3857,
    )

    arr_cmap = plt.cm.get_cmap(cmap).copy()
    arr_cmap.set_bad((1, 1, 1, 0))
    ax.imshow(
        arr,
        cmap=arr_cmap,
        norm=norm,
        alpha=alpha,
        interpolation="nearest",
        origin="upper",
        extent=extent_3857,
    )

    set_axis_bounds(ax, bounds_3857)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    print(filename)

print("Reading terrain rasters for plotting...")
elev_arr, elev_extent_3857 = read_buffered_downsampled(
    terrain_outputs["elevation"],
    max_dim=1400,
)
hs = hillshade(elev_arr)
target_hw = elev_arr.shape

raster_feature_specs = [
    ("elevation", "inferno", None, 0.55),
    ("slope_deg", "inferno", None, 0.60),
    ("tri", "inferno", None, 0.60),
    ("plan_curvature", "RdBu_r", 99, 0.80),
    ("profile_curvature", "RdBu_r", 99, 0.80),
    ("tpi_small", "RdBu_r", 98, 0.80),
    ("tpi_large", "RdBu_r", 98, 0.80),
]

for feature_key, cmap, pct, alpha in raster_feature_specs:
    arr, extent_3857 = read_buffered_downsampled(
        terrain_outputs[feature_key],
        out_hw=target_hw,
        resampling=Resampling.bilinear,
    )

    norm = None
    if pct is not None:
        norm = robust_diverging_norm(arr, pct=pct)

    save_hillshaded_raster(
        arr,
        extent_3857,
        hs,
        OUTPUT_DIR / f"palisades_{feature_key}.png",
        plot_bounds_3857,
        cmap=cmap,
        norm=norm,
        alpha=alpha,
    )

  50/534
  100/534
  150/534
  200/534
  250/534
  300/534
  350/534
  400/534
  450/534
  500/534
  534/534
Merging DEM tiles...
Resampling DEM...
Computing derived terrain rasters with WhiteboxTools...
Decompressing WhiteboxTools_linux_musl.zip ...
WhiteboxTools package directory: /usr/local/lib/python3.12/dist-packages/whitebox
Reading terrain rasters for plotting...


palisades_visual_outputs/palisades_elevation.png


In [9]:
EVT_ZIP_URL = "https://www.landfire.gov/data-downloads/CONUS_LF2024/LF2024_EVT_CONUS.zip"

save_plain_contextily(
    evt_bounds_3857,
    OUTPUT_DIR / "evt_trio_plain_contextily.png",
)

if "VALUE_evt" not in ghex_3857.columns:
    raise KeyError("VALUE_evt not found in the context table.")

evt_extract_dir = ensure_zip_extracted(EVT_ZIP_URL, landfire_dir)
evt_legend_csv = find_landfire_legend_csv(evt_extract_dir)
evt_legend = read_landfire_legend(evt_legend_csv)

save_official_landfire_hex_map(
    ghex_3857,
    "VALUE_evt",
    evt_legend,
    OUTPUT_DIR / "evt_trio_h3_10_hex.png",
    evt_bounds_3857,
)

evt_tif_path = find_first_file(evt_extract_dir, ".tif")
evt_rgb_lookup = build_rgb_lookup_from_legend(evt_legend)

with rasterio.open(evt_tif_path) as src:
    bounds_src = transform_bounds("EPSG:4326", src.crs, *buffered_fetch_bounds_4326, densify_pts=21)
    window = from_bounds(*bounds_src, transform=src.transform)
    window = window.intersection(Window(0, 0, src.width, src.height))

    arr = src.read(1, window=window, masked=True)
    transform = src.window_transform(window)
    raster_crs = src.crs

rows, cols = arr.shape
rgba = np.zeros((rows, cols, 4), dtype=np.uint8)
rgba[..., 3] = 0

valid_mask = ~arr.mask
values = arr.data[valid_mask]

for value in np.unique(values):
    if int(value) not in evt_rgb_lookup:
        continue
    rgb = evt_rgb_lookup[int(value)]
    rgb255 = tuple(int(round(255 * c)) for c in rgb)
    value_mask = valid_mask & (arr.data == value)
    rgba[value_mask, 0] = rgb255[0]
    rgba[value_mask, 1] = rgb255[1]
    rgba[value_mask, 2] = rgb255[2]
    rgba[value_mask, 3] = 255

left, bottom, right, top = rasterio.transform.array_bounds(rows, cols, transform)
extent_3857 = transform_bounds(raster_crs, "EPSG:3857", left, bottom, right, top, densify_pts=21)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(rgba, extent=extent_3857, origin="upper", interpolation="nearest")
set_axis_bounds(ax, evt_bounds_3857)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "evt_trio_raster.png", dpi=300, bbox_inches="tight", pad_inches=0)
plt.close(fig)
print(OUTPUT_DIR / "evt_trio_raster.png")